# EMR on EKS Job Submission Demo

This notebook demonstrates how to use the EMR job submission utilities to submit, monitor, and manage fraud detection jobs on EMR on EKS.

**Features:**
- Submit feature engineering jobs
- Submit training jobs
- Submit inference jobs
- Monitor job status and progress
- List and manage running jobs

In [ ]:
# Import the EMR utilities
import sys
import os

# Add src directory to path
sys.path.append('/home/jovyan/src')

from emr_eks_utils import (
    create_job_manager,
    quick_submit_feature_engineering,
    quick_submit_training,
    monitor_job,
    print_job_status,
    list_recent_jobs,
    EMROnEKSJobManager
)

import boto3
import time
from datetime import datetime

## Environment Setup

In [ ]:
# Check environment variables
required_env_vars = [
    'VIRTUAL_CLUSTER_ID',
    'EMR_EXECUTION_ROLE_ARN',
    'AWS_DEFAULT_REGION',
    'S3_BUCKET'
]

print("Environment Configuration:")
for var in required_env_vars:
    value = os.environ.get(var, 'NOT SET')
    print(f"  {var}: {value}")

# Verify we can create a job manager
try:
    job_manager = create_job_manager()
    print("\n✅ EMR Job Manager created successfully")
except Exception as e:
    print(f"\n❌ Failed to create job manager: {str(e)}")

## List Recent Jobs

In [ ]:
# List recent EMR on EKS jobs
list_recent_jobs(count=10)

## Submit Feature Engineering Job

In [ ]:
# Method 1: Quick submit with defaults
print("Submitting feature engineering job with defaults...")

try:
    fe_job_id = quick_submit_feature_engineering()
    print(f"✅ Feature engineering job submitted: {fe_job_id}")
    
    # Store job ID for monitoring
    feature_engineering_job_id = fe_job_id
    
except Exception as e:
    print(f"❌ Failed to submit job: {str(e)}")

In [ ]:
# Method 2: Submit with custom configuration
print("Submitting feature engineering job with custom configuration...")

try:
    job_manager = create_job_manager()
    
    # Custom Spark configurations
    custom_configs = {
        'spark.sql.shuffle.partitions': '15000',  # Reduce partitions for smaller dataset
        'spark.executor.instances': '8',          # Use fewer executors
        'spark.rapids.memory.pinnedPool.size': '1G'  # Reduce GPU memory pool
    }
    
    # Custom output path
    output_path = f"s3://{os.environ.get('S3_BUCKET')}/fraud-data/custom-features-{datetime.now().strftime('%Y%m%d-%H%M%S')}/"
    
    custom_fe_job_id = job_manager.submit_feature_engineering_job(
        output_path=output_path,
        custom_configs=custom_configs
    )
    
    print(f"✅ Custom feature engineering job submitted: {custom_fe_job_id}")
    print(f"Output path: {output_path}")
    
except Exception as e:
    print(f"❌ Failed to submit custom job: {str(e)}")

## Monitor Job Progress

In [ ]:
# Monitor the feature engineering job
if 'feature_engineering_job_id' in locals():
    print_job_status(feature_engineering_job_id)
else:
    print("No feature engineering job ID available for monitoring")

In [ ]:
# Wait for job completion (optional - this will block until job completes)
# Uncomment the following lines if you want to wait for job completion

# if 'feature_engineering_job_id' in locals():
#     print(f"Waiting for job completion: {feature_engineering_job_id}")
#     final_status = monitor_job(feature_engineering_job_id, wait_for_completion=True)
#     print(f"Final job state: {final_status['state']}")

## Submit Training Job

In [ ]:
# Submit XGBoost training job
print("Submitting XGBoost training job...")

try:
    # Use default paths or specify custom ones
    features_path = f"s3://{os.environ.get('S3_BUCKET')}/fraud-data/processed-features/"
    model_output_path = f"s3://{os.environ.get('S3_BUCKET')}/fraud-models/xgboost-{datetime.now().strftime('%Y%m%d-%H%M%S')}/"
    
    training_job_id = quick_submit_training(
        features_path=features_path,
        model_output_path=model_output_path
    )
    
    print(f"✅ Training job submitted: {training_job_id}")
    print(f"Features path: {features_path}")
    print(f"Model output path: {model_output_path}")
    
except Exception as e:
    print(f"❌ Failed to submit training job: {str(e)}")

## Submit Inference Job

In [ ]:
# Submit batch inference job
print("Submitting batch inference job...")

try:
    job_manager = create_job_manager()
    
    # Specify paths for inference
    model_path = f"s3://{os.environ.get('S3_BUCKET')}/fraud-models/xgboost/"  # Use latest model
    input_data_path = "s3://nvidia-aws-fraud-detection-demo-training-data/transactions_parquet/"  # New transactions
    predictions_output_path = f"s3://{os.environ.get('S3_BUCKET')}/fraud-predictions/batch-{datetime.now().strftime('%Y%m%d-%H%M%S')}/"
    
    inference_job_id = job_manager.submit_inference_job(
        model_path=model_path,
        input_data_path=input_data_path,
        predictions_output_path=predictions_output_path
    )
    
    print(f"✅ Inference job submitted: {inference_job_id}")
    print(f"Model path: {model_path}")
    print(f"Input data path: {input_data_path}")
    print(f"Predictions output path: {predictions_output_path}")
    
except Exception as e:
    print(f"❌ Failed to submit inference job: {str(e)}")

## Advanced Job Management

In [ ]:
# List all running jobs
print("Current running jobs:")
try:
    job_manager = create_job_manager()
    running_jobs = job_manager.list_jobs(states=['RUNNING', 'PENDING'])
    
    if running_jobs:
        for job in running_jobs:
            print(f"  {job['id']} - {job['name']} ({job['state']})")
    else:
        print("  No running jobs found")
        
except Exception as e:
    print(f"Error listing running jobs: {str(e)}")

In [ ]:
# Get detailed job information
job_id_to_check = input("Enter job ID to check (or press Enter to skip): ").strip()

if job_id_to_check:
    try:
        job_manager = create_job_manager()
        status = job_manager.get_job_status(job_id_to_check)
        
        print(f"\n=== Detailed Job Information ===")
        print(f"ID: {status['id']}")
        print(f"Name: {status['name']}")
        print(f"State: {status['state']}")
        print(f"Created: {status['created_at']}")
        print(f"Finished: {status.get('finished_at', 'N/A')}")
        print(f"State Details: {status.get('state_details', 'N/A')}")
        
        if status.get('failure_reason'):
            print(f"Failure Reason: {status['failure_reason']}")
        
        # Get log information
        log_info = job_manager.get_job_logs(job_id_to_check)
        print(f"\nLog Group: {log_info['log_group']}")
        print(f"CloudWatch Console: {log_info['console_url']}")
        
    except Exception as e:
        print(f"Error getting job details: {str(e)}")
else:
    print("Skipping detailed job check")

In [ ]:
# Cancel a job (if needed)
job_id_to_cancel = input("Enter job ID to cancel (or press Enter to skip): ").strip()

if job_id_to_cancel:
    confirm = input(f"Are you sure you want to cancel job {job_id_to_cancel}? (yes/no): ").strip().lower()
    
    if confirm == 'yes':
        try:
            job_manager = create_job_manager()
            success = job_manager.cancel_job(job_id_to_cancel)
            
            if success:
                print(f"✅ Job cancellation requested: {job_id_to_cancel}")
            else:
                print(f"❌ Failed to cancel job: {job_id_to_cancel}")
                
        except Exception as e:
            print(f"Error canceling job: {str(e)}")
    else:
        print("Job cancellation aborted")
else:
    print("Skipping job cancellation")

## Job Templates and Configuration

In [ ]:
# Display available job templates
job_manager = create_job_manager()

print("Available Job Templates:")
for job_type, template in job_manager.job_templates.items():
    print(f"\n{job_type.upper()}:")
    print(f"  Name Prefix: {template['name_prefix']}")
    print(f"  Release Label: {template['release_label']}")
    print(f"  Executor Instances: {template['executor_instances']}")
    print(f"  Executor Memory: {template['executor_memory']}")
    print(f"  GPU Amount: {template['gpu_amount']}")
    print(f"  Key Spark Configs:")
    for key, value in list(template['spark_configs'].items())[:3]:
        print(f"    {key}: {value}")
    if len(template['spark_configs']) > 3:
        print(f"    ... and {len(template['spark_configs']) - 3} more")

## Summary and Next Steps

In [ ]:
print("\n=== EMR on EKS Job Submission Demo Summary ===")
print("\n✅ Capabilities Demonstrated:")
print("  - Job submission with templates (feature engineering, training, inference)")
print("  - Job monitoring and status checking")
print("  - Custom configuration support")
print("  - Job listing and management")
print("  - Log access and troubleshooting")

print("\n📋 Next Steps:")
print("  1. Monitor submitted jobs in EMR console")
print("  2. Check CloudWatch logs for detailed execution information")
print("  3. Verify output data in S3 buckets")
print("  4. Use these utilities in your fraud detection notebooks")
print("  5. Customize job templates for your specific requirements")

print("\n🔗 Useful Commands:")
print("  - AWS CLI: aws emr-containers list-job-runs --virtual-cluster-id <cluster-id>")
print("  - Kubectl: kubectl get pods -n emr-fraud-detection")
print("  - S3: aws s3 ls s3://your-bucket/fraud-data/")

print("\n📚 Documentation:")
print("  - EMR on EKS: https://docs.aws.amazon.com/emr/latest/EMR-on-EKS-DevelopmentGuide/")
print("  - RAPIDS on EMR: https://docs.rapids.ai/")
print("  - Spark Configuration: https://spark.apache.org/docs/latest/configuration.html")